# LJ Dev Commerce - Phase 4

# XIII. Inventory Movement ETL

**Dataset:** `zoho_inventory_movements.csv`  
**Target:** `commerce.inventory_movement`


# =========================================================
# 1.1 Setup
# =========================================================

This notebook performs the approved ETL workflow for the Zoho Inventory Movement source dataset.

Workflow: Raw Dataset → Clean CSV → Database-Ready Dataset → PostgreSQL.

The raw Inventory Movement dataset is never modified directly.

**Known approved exception:** `INM026` references `InventoryRecordID = IN020`, while the completed Inventory parent dataset contains `IN001`–`IN019`. The source record is retained unchanged and is not silently reassigned or deleted. This exception remains visible during relationship/load validation.


In [1]:
# =========================================================
# 1.1 Setup
# =========================================================

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)


In [ ]:
# Project path handling is configured in the dataset load cell.


## 1.2 Load and Preserve Inventory Movement Raw Dataset

Load the untouched Zoho Inventory Movement source dataset and preserve an original copy for validation.


In [2]:
# =========================================================
# 1.2 Load and Preserve Inventory Movement Raw Dataset
# =========================================================

inventory_movement_raw_path = Path(
    r"C:\JEP\DATA ANALYST PORTFOLIO\ecommerce-data-analytics"
    r"\data\04_Zoho_Inventory\zoho_inventory_movements.csv"
)

inventory_movement_raw = pd.read_csv(inventory_movement_raw_path)
inventory_movement_raw_original = inventory_movement_raw.copy(deep=True)

print("Inventory Movement raw dataset loaded successfully.")
print("Rows:", len(inventory_movement_raw))
print("Columns:", len(inventory_movement_raw.columns))
print("Path:", inventory_movement_raw_path)


Inventory Movement raw dataset loaded successfully.
Rows: 26
Columns: 11
Path: C:\JEP\DATA ANALYST PORTFOLIO\ecommerce-data-analytics\data\04_Zoho_Inventory\zoho_inventory_movements.csv


## 1.3 Initial Raw Data Inspection

Inspect the untouched Inventory Movement dataset before any cleaning or transformation.


In [3]:
# =========================================================
# 1.3 Initial Raw Data Inspection
# =========================================================

print("Inventory Movement Dataset shape:")
print(inventory_movement_raw.shape)
print("\nInventory Column names:")
print(inventory_movement_raw.columns.tolist())
print("\nInventory Movement pandas data types:")
print(inventory_movement_raw.dtypes)
print("\nInventory Missing values:")
print(inventory_movement_raw.isnull().sum())
print("\nInventory Sample raw records:")
display(inventory_movement_raw.head())


Inventory Movement Dataset shape:
(26, 11)

Inventory Column names:
['MovementID', 'InventoryRecordID', 'MovementDate', 'MovementType', 'QuantityChange', 'ReferenceType', 'ReferenceID', 'CreatedOn', 'CreatedByUser', 'ModifiedOn', 'ModifiedByUser']

Inventory Movement pandas data types:
MovementID           object
InventoryRecordID    object
MovementDate         object
MovementType         object
QuantityChange        int64
ReferenceType        object
ReferenceID          object
CreatedOn            object
CreatedByUser        object
ModifiedOn           object
ModifiedByUser       object
dtype: object

Inventory Missing values:
MovementID           0
InventoryRecordID    0
MovementDate         0
MovementType         0
QuantityChange       0
ReferenceType        0
ReferenceID          0
CreatedOn            0
CreatedByUser        0
ModifiedOn           0
ModifiedByUser       0
dtype: int64

Inventory Sample raw records:


,MovementID,InventoryRecordID,MovementDate,MovementType,QuantityChange,ReferenceType,ReferenceID,CreatedOn,CreatedByUser,ModifiedOn,ModifiedByUser
0,INM001,IN001,2026-07-05,Purchase,20,Purchase Order,PO001,2026-07-05 12:00:00,admin,2026-07-05 13:00:00,admin
1,INM002,IN001,2026-07-08,Sale,-1,Sales Order,SO001,2026-07-08 12:00:00,admin,2026-07-08 13:00:00,admin
2,INM003,IN001,2026-07-12,Sale,-1,Sales Order,SO016,2026-07-12 12:00:00,admin,2026-07-12 13:00:00,admin
3,INM004,IN002,2026-07-06,Purchase,35,Purchase Order,PO002,2026-07-06 12:00:00,admin,2026-07-06 13:00:00,admin
4,INM005,IN002,2026-07-09,Sale,-1,Sales Order,SO002,2026-07-09 12:00:00,admin,2026-07-09 13:00:00,admin


## 1.4 Data Profiling

Profile the untouched Inventory Movement dataset using the reusable project profiler.


In [4]:
# =========================================================
# 1.4 Data Profiling
# =========================================================

import sys
import importlib

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from profiler import data_profiler_v1 as profiler
importlib.reload(profiler)

print("Profiler loaded successfully:")
print(profiler.__file__)
print("\nProfiler configuration:")
print(profiler.DEFAULT_CONFIG)


Profiler loaded successfully:
c:\JEP\DATA ANALYST PORTFOLIO\ecommerce-data-analytics\profiler\data_profiler_v1.py

Profiler configuration:
{'date_detection_threshold': 0.8, 'numeric_detection_threshold': 0.8, 'email_detection_threshold': 0.8, 'phone_detection_threshold': 0.8, 'categorical_unique_ratio': 0.2, 'outlier_iqr_multiplier': 1.5, 'required_columns': [], 'unique_columns': [], 'non_negative_columns': []}


In [5]:
inventory_movement_profile_results = profiler.profile_dataset(inventory_movement_raw)
print("\nInventory profiling completed successfully.")
print("\nInventory profiler sections:")
print(list(inventory_movement_profile_results.keys()))



Inventory profiling completed successfully.

Inventory profiler sections:
['field_types', 'detection_details', 'general', 'text', 'categorical', 'numeric', 'date', 'patterns', 'issues', 'configuration']


## 1.5 Review Profiling Results

Review the complete Inventory profiling output before making transformation decisions.


In [6]:
print("\n" + "=" * 70)
print("INVENTORY PROFILING RESULTS")
print("=" * 70)
for section_name, section_result in inventory_movement_profile_results.items():
    print("\n" + "-" * 70)
    print(section_name.upper())
    print("-" * 70)
    print(section_result)



INVENTORY PROFILING RESULTS

----------------------------------------------------------------------
FIELD_TYPES
----------------------------------------------------------------------
MovementID            identifier
InventoryRecordID     identifier
MovementDate                date
MovementType         categorical
QuantityChange           numeric
ReferenceType         identifier
ReferenceID           identifier
CreatedOn                   date
CreatedByUser        categorical
ModifiedOn                  date
ModifiedByUser       categorical
Name: DetectedFieldType, dtype: object

----------------------------------------------------------------------
DETECTION_DETAILS
----------------------------------------------------------------------
{'MovementID': {'DetectedType': 'identifier', 'Confidence': 'Medium', 'Evidence': 'Column name suggests identifier semantics'}, 'InventoryRecordID': {'DetectedType': 'identifier', 'Confidence': 'Medium', 'Evidence': 'Column name suggests identifier sema

## 1.6 Review Profiler Issues

Review the profiler's detected Inventory issues separately and classify them before transformation.


In [7]:
inventory_movement_issues = inventory_movement_profile_results["issues"]
print("\n" + "=" * 70)
print("INVENTORY PROFILER ISSUES")
print("=" * 70)
display(inventory_movement_issues)



INVENTORY PROFILER ISSUES


,Column,IssueType,Severity,Count,Description
0,InventoryRecordID,DuplicateIdentifier,High,6,Repeated identifier values detected.
1,ReferenceType,DuplicateIdentifier,High,22,Repeated identifier values detected.


## 1.7 Independent Analyst Review

Perform the systematic independent analyst review required by the locked ETL workflow.


In [8]:
# =========================================================
# 1.7 Independent Analyst Review
# =========================================================

source_df = inventory_movement_raw

print("1. DATASET STRUCTURE AND GRAIN")
print("=" * 80)
print(f"Rows: {source_df.shape[0]}")
print(f"Columns: {source_df.shape[1]}")
print("Working assumption: One row represents one inventory movement transaction.")

print("\n\n2. MISSING VALUES AND BLANK VALUES")
missing_values = source_df.isna().sum()
blank_values = pd.Series({c: (source_df[c].astype("string").str.strip().eq("").sum() if source_df[c].dtype == "object" else 0) for c in source_df.columns})
display(pd.DataFrame({"MissingValues": missing_values, "BlankValues": blank_values}))

print("\n\n3. EXACT DUPLICATE ROWS")
exact_duplicate_count = source_df.duplicated().sum()
print("Exact duplicate rows:", exact_duplicate_count)
if exact_duplicate_count > 0:
    display(source_df[source_df.duplicated(keep=False)])

print("\n\n4. MOVEMENT IDENTIFIER REVIEW")
identifier_columns = ["MovementID", "InventoryRecordID", "ReferenceID"]
identifier_review = pd.DataFrame({
    "Column": identifier_columns,
    "NullCount": [source_df[c].isna().sum() for c in identifier_columns],
    "BlankCount": [source_df[c].astype("string").str.strip().eq("").sum() for c in identifier_columns],
    "DuplicateCount": [source_df[c].duplicated().sum() for c in identifier_columns],
    "UniqueValues": [source_df[c].nunique(dropna=True) for c in identifier_columns]
})
display(identifier_review)

print("\n\n5. MULTIPLE MOVEMENTS PER INVENTORY RECORD")
movement_counts = source_df["InventoryRecordID"].value_counts().sort_index()
display(movement_counts.rename("MovementCount").to_frame())
print("Multiple movements per InventoryRecordID are expected for historical Inventory Movement data.")

print("\n\n6. MOVEMENT TYPE / REFERENCE TYPE CONSISTENCY")
movement_reference_review = pd.crosstab(source_df["MovementType"], source_df["ReferenceType"])
display(movement_reference_review)
expected_pairs = {("Purchase", "Purchase Order"), ("Sale", "Sales Order"), ("Return", "Return"), ("Adjustment", "Adjustment")}
observed_pairs = set(zip(source_df["MovementType"], source_df["ReferenceType"]))
print("Observed pairs:", observed_pairs)
print("All observed movement/reference combinations are approved patterns:", observed_pairs.issubset(expected_pairs))

print("\n\n7. QUANTITY CHANGE SIGN REVIEW")
for movement_type, group in source_df.groupby("MovementType"):
    print(f"{movement_type}: rows={len(group)}, min={group['QuantityChange'].min()}, max={group['QuantityChange'].max()}")

print("\n\n8. DATE VALIDITY AND DATE RELATIONSHIPS")
parsed_movement = pd.to_datetime(source_df["MovementDate"], errors="coerce")
parsed_created = pd.to_datetime(source_df["CreatedOn"], errors="coerce")
parsed_modified = pd.to_datetime(source_df["ModifiedOn"], errors="coerce")
print("Invalid MovementDate values:", int(parsed_movement.isna().sum()))
print("Invalid CreatedOn values:", int(parsed_created.isna().sum()))
print("Invalid ModifiedOn values:", int(parsed_modified.isna().sum()))
print("MovementDate <= CreatedOn <= ModifiedOn for all rows:", bool((parsed_movement <= parsed_created).all() and (parsed_created <= parsed_modified).all()))

print("\n\n9. TEXT / IDENTIFIER WHITESPACE REVIEW")
whitespace_findings=[]
for column in source_df.select_dtypes(include="object").columns:
    mask=source_df[column].astype("string") != source_df[column].astype("string").str.strip()
    if int(mask.sum()) > 0:
        whitespace_findings.append({"Column":column,"WhitespaceRecordCount":int(mask.sum())})
if whitespace_findings:
    display(pd.DataFrame(whitespace_findings))
else:
    print("No leading or trailing whitespace found.")

print("\n\n10. KNOWN REFERENTIAL INTEGRITY EXCEPTION")
inm026 = source_df[source_df["MovementID"] == "INM026"]
display(inm026)
print("INM026 references InventoryRecordID IN020.")
print("Completed Inventory source contains IN001 through IN019 only.")
print("Decision: retain INM026 unchanged; do not replace its parent reference and do not delete the historical movement.")


1. DATASET STRUCTURE AND GRAIN
Rows: 26
Columns: 11
Working assumption: One row represents one inventory movement transaction.


2. MISSING VALUES AND BLANK VALUES


,MissingValues,BlankValues
MovementID,0,0
InventoryRecordID,0,0
MovementDate,0,0
MovementType,0,0
QuantityChange,0,0
ReferenceType,0,0
ReferenceID,0,0
CreatedOn,0,0
CreatedByUser,0,0
ModifiedOn,0,0




3. EXACT DUPLICATE ROWS
Exact duplicate rows: 0


4. MOVEMENT IDENTIFIER REVIEW


,Column,NullCount,BlankCount,DuplicateCount,UniqueValues
0,MovementID,0,0,0,26
1,InventoryRecordID,0,0,6,20
2,ReferenceID,0,0,0,26




5. MULTIPLE MOVEMENTS PER INVENTORY RECORD


...

Multiple movements per InventoryRecordID are expected for historical Inventory Movement data.


6. MOVEMENT TYPE / REFERENCE TYPE CONSISTENCY


ReferenceType,Adjustment,Purchase Order,Return,Sales Order
MovementType,,,,
Adjustment,1,0,0,0
Purchase,0,16,0,0
Return,0,0,1,0
Sale,0,0,0,8


Observed pairs: {('Return', 'Return'), ('Sale', 'Sales Order'), ('Adjustment', 'Adjustment'), ('Purchase', 'Purchase Order')}
All observed movement/reference combinations are approved patterns: True


7. QUANTITY CHANGE SIGN REVIEW
Adjustment: rows=1, min=-2, max=-2
Purchase: rows=16, min=10, max=65
Return: rows=1, min=1, max=1
Sale: rows=8, min=-1, max=-1


8. DATE VALIDITY AND DATE RELATIONSHIPS
Invalid MovementDate values: 0
Invalid CreatedOn values: 0
Invalid ModifiedOn values: 0
MovementDate <= CreatedOn <= ModifiedOn for all rows: True


9. TEXT / IDENTIFIER WHITESPACE REVIEW
No leading or trailing whitespace found.


10. KNOWN REFERENTIAL INTEGRITY EXCEPTION


,MovementID,InventoryRecordID,MovementDate,MovementType,QuantityChange,ReferenceType,ReferenceID,CreatedOn,CreatedByUser,ModifiedOn,ModifiedByUser
25,INM026,IN020,2026-07-20,Sale,-1,Sales Order,SO013,2026-07-20 12:00:00,admin,2026-07-20 13:00:00,admin


INM026 references InventoryRecordID IN020.
Completed Inventory source contains IN001 through IN019 only.
Decision: retain INM026 unchanged; do not replace its parent reference and do not delete the historical movement.


## 1.8 Findings and Transformation Decisions

Compare profiler findings, independent analyst findings, the approved Inventory Movement Data Dictionary, and the completed pre-ETL review. Only approved transformations proceed to Section 2.

The unresolved `INM026 → IN020` parent-reference issue is not a transformation. It remains a relationship/load dependency exception and must not be silently corrected.


In [9]:
inventory_movement_transformation_decisions = pd.DataFrame([
["MovementID uniqueness",26,"Valid value — retain unchanged","Use as inventory_movement_id","Required primary identifier."],
["InventoryRecordID values",26,"Retain source values; relationship validated later","Map to inventory_id unchanged","Required foreign key. INM026 references unresolved IN020 and must not be reassigned."],
["MovementDate object dtype",26,"Safe deterministic datatype transformation","Convert to datetime","Target movement_date is Datetime and Required."],
["CreatedOn object dtype",26,"Safe deterministic datatype transformation","Convert to datetime","Target created_date is Datetime and Required."],
["ModifiedOn object dtype",26,"Safe deterministic datatype transformation","Convert to datetime","Target updated_date is Datetime and Required."],
["MovementType / ReferenceType combinations",26,"Valid observed source values","Keep values unchanged","Observed combinations are consistent with approved movement/reference patterns."],
["QuantityChange signs",26,"Valid source business values","Keep values unchanged","Purchase/Return increases and Sale/Adjustment decreases are preserved."],
["ReferenceID values",26,"Valid source identifiers","Keep values unchanged","Reference identifiers are retained for audit traceability."],
["Audit fields",26,"Valid source values","Keep values unchanged","No approved cleaning requirement identified."],
["INM026 → IN020 relationship",1,"Unresolved referential-integrity exception","Retain unchanged and document; block database load until parent is resolved","No supported basis exists to map IN020 to another InventoryRecordID."],
],columns=["Finding","AffectedRecords","Decision","ApprovedAction","Reason"])
display(inventory_movement_transformation_decisions)
print("Inventory Movement transformation decisions recorded.")
print("Transformation status: READY")
print("Database load status: PENDING due to unresolved INM026 → IN020 parent reference.")


,Finding,AffectedRecords,Decision,ApprovedAction,Reason
0,MovementID uniqueness,26,Valid value — retain unchanged,Use as inventory_movement_id,Required primary identifier.
1,InventoryRecordID values,26,Retain source values; relationship validated later,Map to inventory_id unchanged,Required foreign key. INM026 references unresolved IN020 and must not be reassigned.
2,MovementDate object dtype,26,Safe deterministic datatype transformation,Convert to datetime,Target movement_date is Datetime and Required.
3,CreatedOn object dtype,26,Safe deterministic datatype transformation,Convert to datetime,Target created_date is Datetime and Required.
4,ModifiedOn object dtype,26,Safe deterministic datatype transformation,Convert to datetime,Target updated_date is Datetime and Required.
5,MovementType / ReferenceType combinations,26,Valid observed source values,Keep values unchanged,Observed combinations are consistent with approved movement/reference patterns.
6,QuantityChange signs,26,Valid source business values,Keep values unchanged,Purchase/Return increases and Sale/Adjustment decreases are preserved.
7,ReferenceID values,26,Valid source identifiers,Keep values unchanged,Reference identifiers are retained for audit traceability.
8,Audit fields,26,Valid source values,Keep values unchanged,No approved cleaning requirement identified.
9,INM026 → IN020 relationship,1,Unresolved referential-integrity exception,Retain unchanged and document; block database load until parent is resolved,No supported basis exists to map IN020 to another InventoryRecordID.


Inventory Movement transformation decisions recorded.
Transformation status: READY
Database load status: PENDING due to unresolved INM026 → IN020 parent reference.


# =========================================================
# SECTION 2 — DATA TRANSFORMATION
# =========================================================

## 2.1 Create Clean Working Dataset

Create a separate working copy of the raw Inventory Movement dataset before applying any approved transformations.

The raw Inventory Movement dataset must remain unchanged throughout the ETL process.

All transformations will be applied only to the clean working dataset.

The Inventory Movement source dataset will remain as a single working dataset during this step.


In [10]:
inventory_movement_clean = inventory_movement_raw.copy(deep=True)
print("Inventory Movement Clean DataFrame created.")
print("Rows:", len(inventory_movement_clean))
print("Columns:", len(inventory_movement_clean.columns))


Inventory Movement Clean DataFrame created.
Rows: 26
Columns: 11


## 2.2 Apply Approved Transformations

Apply only the approved Inventory transformations documented in Section 1.8.


In [11]:
inventory_movement_clean["MovementDate"] = pd.to_datetime(inventory_movement_clean["MovementDate"], errors="raise")
for column in ["CreatedOn", "ModifiedOn"]:
    inventory_movement_clean[column] = pd.to_datetime(inventory_movement_clean[column], errors="raise")
print("Approved Inventory Movement transformations applied.")
print("No identifier, quantity, movement type, reference type, or reference ID values were changed.")


Approved Inventory Movement transformations applied.
No identifier, quantity, movement type, reference type, or reference ID values were changed.


## 3.1 Transformation Validation

Validate every approved Inventory transformation from Section 1.8.

Checks include StockStatus standardization, datetime conversion, preservation of approved quantity relationships, and preservation of all other approved Inventory values.


In [12]:
datetime_valid = all(pd.api.types.is_datetime64_any_dtype(inventory_movement_clean[c]) for c in ["MovementDate","CreatedOn","ModifiedOn"])
identifier_values_preserved = all(inventory_movement_clean[c].tolist() == inventory_movement_raw[c].tolist() for c in ["MovementID","InventoryRecordID","ReferenceID"])
quantity_values_preserved = inventory_movement_clean["QuantityChange"].equals(inventory_movement_raw["QuantityChange"])
movement_reference_values_preserved = all(inventory_movement_clean[c].tolist() == inventory_movement_raw[c].tolist() for c in ["MovementType","ReferenceType"])
inm026_reference_preserved = inventory_movement_clean.loc[inventory_movement_clean["MovementID"] == "INM026","InventoryRecordID"].iloc[0] == "IN020"
transformation_validation_passed = all([datetime_valid,identifier_values_preserved,quantity_values_preserved,movement_reference_values_preserved,inm026_reference_preserved])
print("Inventory Movement Transformation Validation")
print("Datetime fields valid:", datetime_valid)
print("Identifiers preserved:", identifier_values_preserved)
print("Quantity values preserved:", quantity_values_preserved)
print("Movement/reference values preserved:", movement_reference_values_preserved)
print("INM026 → IN020 source relationship preserved:", inm026_reference_preserved)
print("Overall transformation validation passed:", transformation_validation_passed)
assert transformation_validation_passed


Inventory Movement Transformation Validation
Datetime fields valid: True
Identifiers preserved: True
Quantity values preserved: True
Movement/reference values preserved: True
INM026 → IN020 source relationship preserved: True
Overall transformation validation passed: True


## 3.2 Validate Data Preservation

Validate that the approved Inventory transformations did not unintentionally change unrelated values or remove records.


In [13]:
preserved_columns=["MovementID","InventoryRecordID","MovementType","QuantityChange","ReferenceType","ReferenceID","CreatedByUser","ModifiedByUser"]
row_count_preserved=len(inventory_movement_clean)==len(inventory_movement_raw)
column_structure_preserved=inventory_movement_clean.columns.tolist()==inventory_movement_raw.columns.tolist()
preservation_results={c:inventory_movement_clean[c].equals(inventory_movement_raw[c]) for c in preserved_columns}
preserved_values_valid=all(preservation_results.values())
movement_ids_preserved=inventory_movement_clean["MovementID"].tolist()==inventory_movement_raw["MovementID"].tolist()
data_preservation_passed=all([row_count_preserved,column_structure_preserved,preserved_values_valid,movement_ids_preserved])
display(pd.Series(preservation_results,name="Preserved"))
print("Row count preserved:",row_count_preserved)
print("Column structure preserved:",column_structure_preserved)
print("Movement IDs preserved:",movement_ids_preserved)
print("Unchanged values preserved:",preserved_values_valid)
print("Overall data preservation validation passed:",data_preservation_passed)
assert data_preservation_passed


MovementID           True
InventoryRecordID    True
MovementType         True
QuantityChange       True
ReferenceType        True
ReferenceID          True
CreatedByUser        True
ModifiedByUser       True
Name: Preserved, dtype: bool

Row count preserved: True
Column structure preserved: True
Movement IDs preserved: True
Unchanged values preserved: True
Overall data preservation validation passed: True


# =========================================================
# SECTION 4 — CLEAN CSV
# =========================================================


In [14]:
inventory_movement_clean_dir=Path(r"C:\JEP\DATA ANALYST PORTFOLIO\ecommerce-data-analytics\data\04_Zoho_Inventory\clean")
inventory_movement_clean_dir.mkdir(parents=True,exist_ok=True)
inventory_movement_clean_path=inventory_movement_clean_dir / "zoho_inventory_movements_clean.csv"
inventory_movement_clean.to_csv(inventory_movement_clean_path,index=False)
print("Clean Inventory Movement CSV exported successfully.")
print("Path:",inventory_movement_clean_path)


Clean Inventory Movement CSV exported successfully.
Path: C:\JEP\DATA ANALYST PORTFOLIO\ecommerce-data-analytics\data\04_Zoho_Inventory\clean\zoho_inventory_movements_clean.csv


## 4.2 Verify Exported Clean Inventory Movement CSV

Reload the exported Clean CSV and verify row count and column structure against the validated Clean DataFrame.


In [15]:
inventory_movement_clean_csv=pd.read_csv(inventory_movement_clean_path,parse_dates=["MovementDate","CreatedOn","ModifiedOn"])
row_count_match=len(inventory_movement_clean_csv)==len(inventory_movement_clean)
column_structure_match=inventory_movement_clean_csv.columns.tolist()==inventory_movement_clean.columns.tolist()
print("Inventory Movement Clean CSV read-back successful.")
print("Rows:",len(inventory_movement_clean_csv))
print("Columns:",len(inventory_movement_clean_csv.columns))
print("Row count matches:",row_count_match)
print("Column structure matches:",column_structure_match)
display(inventory_movement_clean_csv.head())
assert row_count_match and column_structure_match


Inventory Movement Clean CSV read-back successful.
Rows: 26
Columns: 11
Row count matches: True
Column structure matches: True


,MovementID,InventoryRecordID,MovementDate,MovementType,QuantityChange,ReferenceType,ReferenceID,CreatedOn,CreatedByUser,ModifiedOn,ModifiedByUser
0,INM001,IN001,2026-07-05,Purchase,20,Purchase Order,PO001,2026-07-05 12:00:00,admin,2026-07-05 13:00:00,admin
1,INM002,IN001,2026-07-08,Sale,-1,Sales Order,SO001,2026-07-08 12:00:00,admin,2026-07-08 13:00:00,admin
2,INM003,IN001,2026-07-12,Sale,-1,Sales Order,SO016,2026-07-12 12:00:00,admin,2026-07-12 13:00:00,admin
3,INM004,IN002,2026-07-06,Purchase,35,Purchase Order,PO002,2026-07-06 12:00:00,admin,2026-07-06 13:00:00,admin
4,INM005,IN002,2026-07-09,Sale,-1,Sales Order,SO002,2026-07-09 12:00:00,admin,2026-07-09 13:00:00,admin


# =========================================================
# SECTION 5 — DATABASE READY
# =========================================================


In [16]:
inventory_movement_mapping=pd.DataFrame([
["MovementID","inventory_movement_id","TEXT","Required","PK",""],
["InventoryRecordID","inventory_id","TEXT","Required","FK","commerce.inventory.inventory_id"],
["MovementDate","movement_date","TIMESTAMP","Required","",""],
["MovementType","movement_type","TEXT","Required","",""],
["QuantityChange","quantity_change","INTEGER","Required","",""],
["ReferenceType","reference_type","TEXT","Required","",""],
["ReferenceID","reference_id","TEXT","Required","",""],
["CreatedOn","created_date","TIMESTAMP","Required","",""],
["CreatedByUser","created_by","TEXT","Required","",""],
["ModifiedOn","updated_date","TIMESTAMP","Required","",""],
["ModifiedByUser","updated_by","TEXT","Required","",""],
],columns=["Source Column","Target Column","Target Datatype","Required","Key Role","Reference / Rule"])
display(inventory_movement_mapping)


,Source Column,Target Column,Target Datatype,Required,Key Role,Reference / Rule
0,MovementID,inventory_movement_id,TEXT,Required,PK,
1,InventoryRecordID,inventory_id,TEXT,Required,FK,commerce.inventory.inventory_id
2,MovementDate,movement_date,TIMESTAMP,Required,,
3,MovementType,movement_type,TEXT,Required,,
4,QuantityChange,quantity_change,INTEGER,Required,,
5,ReferenceType,reference_type,TEXT,Required,,
6,ReferenceID,reference_id,TEXT,Required,,
7,CreatedOn,created_date,TIMESTAMP,Required,,
8,CreatedByUser,created_by,TEXT,Required,,
9,ModifiedOn,updated_date,TIMESTAMP,Required,,


## 5.2 Create Database-Ready Inventory Movement Dataset

Create the Database-Ready Inventory Movement dataset from the verified exported Clean CSV, not directly from the in-memory Clean DataFrame.


In [17]:
inventory_movement_db_source=pd.read_csv(inventory_movement_clean_path,parse_dates=["MovementDate","CreatedOn","ModifiedOn"])
inventory_movement_db_ready=inventory_movement_db_source.rename(columns={
"MovementID":"inventory_movement_id","InventoryRecordID":"inventory_id","MovementDate":"movement_date","MovementType":"movement_type","QuantityChange":"quantity_change","ReferenceType":"reference_type","ReferenceID":"reference_id","CreatedOn":"created_date","CreatedByUser":"created_by","ModifiedOn":"updated_date","ModifiedByUser":"updated_by"})[["inventory_movement_id","inventory_id","movement_date","movement_type","quantity_change","reference_type","reference_id","created_date","created_by","updated_date","updated_by"]].copy()
print("Inventory Movement Database-Ready dataset created from exported Clean CSV.")
print("Rows:",len(inventory_movement_db_ready))
print("Columns:",len(inventory_movement_db_ready.columns))
display(inventory_movement_db_ready.head())


Inventory Movement Database-Ready dataset created from exported Clean CSV.
Rows: 26
Columns: 11


,inventory_movement_id,inventory_id,movement_date,movement_type,quantity_change,reference_type,reference_id,created_date,created_by,updated_date,updated_by
0,INM001,IN001,2026-07-05,Purchase,20,Purchase Order,PO001,2026-07-05 12:00:00,admin,2026-07-05 13:00:00,admin
1,INM002,IN001,2026-07-08,Sale,-1,Sales Order,SO001,2026-07-08 12:00:00,admin,2026-07-08 13:00:00,admin
2,INM003,IN001,2026-07-12,Sale,-1,Sales Order,SO016,2026-07-12 12:00:00,admin,2026-07-12 13:00:00,admin
3,INM004,IN002,2026-07-06,Purchase,35,Purchase Order,PO002,2026-07-06 12:00:00,admin,2026-07-06 13:00:00,admin
4,INM005,IN002,2026-07-09,Sale,-1,Sales Order,SO002,2026-07-09 12:00:00,admin,2026-07-09 13:00:00,admin


## 5.3 Database-Ready Validation

Validate the Inventory Movement Database-Ready dataset against the approved target columns, required fields, primary key, datatypes, mapping completeness, and foreign-key readiness.

The `INM026 → IN020` exception is expected to fail the Inventory parent dependency check. It must be reported, not corrected or silently removed.


In [18]:
expected_target_columns=["inventory_movement_id","inventory_id","movement_date","movement_type","quantity_change","reference_type","reference_id","created_date","created_by","updated_date","updated_by"]
columns_match=inventory_movement_db_ready.columns.tolist()==expected_target_columns
required_fields_valid=all(inventory_movement_db_ready[c].notna().all() for c in expected_target_columns)
primary_key_unique=inventory_movement_db_ready["inventory_movement_id"].is_unique
text_columns=["inventory_movement_id","inventory_id","movement_type","reference_type","reference_id","created_by","updated_by"]
text_datatypes_valid=all(pd.api.types.is_object_dtype(inventory_movement_db_ready[c]) for c in text_columns)
integer_datatype_valid=pd.api.types.is_integer_dtype(inventory_movement_db_ready["quantity_change"])
datetime_datatypes_valid=all(pd.api.types.is_datetime64_any_dtype(inventory_movement_db_ready[c]) for c in ["movement_date","created_date","updated_date"])
inventory_parent_clean_path=inventory_movement_clean_dir / "zoho_inventory_current_stock_clean.csv"
parent_dependency_file_exists=inventory_parent_clean_path.exists()
if parent_dependency_file_exists:
    inventory_parent_df=pd.read_csv(inventory_parent_clean_path)
    valid_inventory_ids=set(inventory_parent_df["InventoryRecordID"].astype(str))
    missing_inventory_ids=sorted(set(inventory_movement_db_ready["inventory_id"].astype(str))-valid_inventory_ids)
else:
    missing_inventory_ids=sorted(inventory_movement_db_ready["inventory_id"].astype(str).unique())
inventory_fk_ready=parent_dependency_file_exists and len(missing_inventory_ids)==0
print("Inventory Movement Database-Ready Validation")
print("Target columns and order match:",columns_match)
print("Required fields valid:",required_fields_valid)
print("Primary key unique:",primary_key_unique)
print("Text datatypes valid:",text_datatypes_valid)
print("Quantity integer datatype valid:",integer_datatype_valid)
print("Datetime datatypes valid:",datetime_datatypes_valid)
print("Inventory parent dependency file available:",parent_dependency_file_exists)
print("Missing Inventory parent IDs:",missing_inventory_ids)
print("Inventory foreign-key readiness:",inventory_fk_ready)
database_ready_structural_validation_passed=all([columns_match,required_fields_valid,primary_key_unique,text_datatypes_valid,integer_datatype_valid,datetime_datatypes_valid])
print("Database-ready structural validation passed:",database_ready_structural_validation_passed)
print("Database load dependency ready:",inventory_fk_ready)


Inventory Movement Database-Ready Validation
Target columns and order match: True
Required fields valid: True
Primary key unique: True
Text datatypes valid: True
Quantity integer datatype valid: True
Datetime datatypes valid: True
Inventory parent dependency file available: True
Missing Inventory parent IDs: ['IN020']
Inventory foreign-key readiness: False
Database-ready structural validation passed: True
Database load dependency ready: False


# =========================================================
# SECTION 6 — PostgreSQL
# =========================================================


## 6.1 Connect

Establish a PostgreSQL connection for the Inventory Movement ETL process.


In [19]:
import psycopg2
from getpass import getpass
DB_HOST="localhost"
DB_PORT="5432"
DB_NAME="lj_dev_commerce"
DB_USER="postgres"
DB_PASSWORD=getpass("Enter PostgreSQL password: ")
conn=None
cursor=None
try:
    conn=psycopg2.connect(host=DB_HOST,port=DB_PORT,dbname=DB_NAME,user=DB_USER,password=DB_PASSWORD)
    cursor=conn.cursor()
    print("PostgreSQL connection successful.")
    print("Database:",DB_NAME)
    print("Host:",DB_HOST)
    print("Port:",DB_PORT)
except Exception as e:
    print("PostgreSQL connection failed.")
    print("Error:",e)


PostgreSQL connection successful.
Database: lj_dev_commerce
Host: localhost
Port: 5432


## 6.2 Prepare / Create Target Table

Check whether the approved `commerce.inventory_movement` target table already exists. If it exists, do not recreate it.


In [20]:
target_schema="commerce"
target_table="inventory_movement"
cursor.execute("SELECT EXISTS (SELECT 1 FROM information_schema.tables WHERE table_schema=%s AND table_name=%s);",(target_schema,target_table))
table_exists=cursor.fetchone()[0]
print("Inventory Movement Target Table Check")
print("Schema:",target_schema)
print("Table:",target_table)
print("Table exists:",table_exists)


Inventory Movement Target Table Check
Schema: commerce
Table: inventory_movement
Table exists: False


## 6.2.1 Create Inventory Movement Table

Create `commerce.inventory_movement` only when the target table does not already exist, using the approved Inventory Movement Data Dictionary and the Inventory foreign-key relationship.

The optional target field `notes` is nullable. The current source extract contains no Notes column, so it is loaded as NULL when records are inserted.


In [21]:
if not table_exists:
    create_inventory_movement_table_query="""
    CREATE TABLE commerce.inventory_movement (
        inventory_movement_id TEXT PRIMARY KEY,
        inventory_id TEXT NOT NULL,
        movement_date TIMESTAMP NOT NULL,
        movement_type TEXT NOT NULL,
        quantity_change INTEGER NOT NULL,
        reference_type TEXT NOT NULL,
        reference_id TEXT NOT NULL,
        notes TEXT,
        created_date TIMESTAMP NOT NULL,
        created_by TEXT NOT NULL,
        updated_date TIMESTAMP NOT NULL,
        updated_by TEXT NOT NULL,
        FOREIGN KEY (inventory_id) REFERENCES commerce.inventory(inventory_id)
    );
    """
    try:
        cursor.execute(create_inventory_movement_table_query)
        conn.commit()
        print("Inventory Movement table created successfully.")
    except Exception as e:
        conn.rollback()
        print("Inventory Movement table creation failed.")
        print("Error:",e)
else:
    print("Inventory Movement table already exists. Creation skipped.")


Inventory Movement table created successfully.


## 6.3 Verify Inventory Movement Table Structure

Verify the actual PostgreSQL structure of `commerce.inventory_movement` against the approved target design.


In [22]:
verify_inventory_movement_table_query="""
SELECT ordinal_position,column_name,data_type,is_nullable,column_default
FROM information_schema.columns
WHERE table_schema='commerce' AND table_name='inventory_movement'
ORDER BY ordinal_position;
"""
try:
    cursor.execute(verify_inventory_movement_table_query)
    table_structure=cursor.fetchall()
    expected_structure=[
    (1,"inventory_movement_id","text","NO"),(2,"inventory_id","text","NO"),(3,"movement_date","timestamp without time zone","NO"),(4,"movement_type","text","NO"),(5,"quantity_change","integer","NO"),(6,"reference_type","text","NO"),(7,"reference_id","text","NO"),(8,"notes","text","YES"),(9,"created_date","timestamp without time zone","NO"),(10,"created_by","text","NO"),(11,"updated_date","timestamp without time zone","NO"),(12,"updated_by","text","NO")]
    actual_structure=[(row[0],row[1],row[2],row[3]) for row in table_structure]
    structure_valid=actual_structure==expected_structure
    print("Inventory Movement Table Structure")
    for row in table_structure: print(row)
    print("Inventory Movement table structure validation passed:",structure_valid)
    assert structure_valid
except Exception as e:
    print("Inventory Movement table structure verification failed.")
    print("Error:",e)
    raise


Inventory Movement Table Structure
(1, 'inventory_movement_id', 'text', 'NO', None)
(2, 'inventory_id', 'text', 'NO', None)
(3, 'movement_date', 'timestamp without time zone', 'NO', None)
(4, 'movement_type', 'text', 'NO', None)
(5, 'quantity_change', 'integer', 'NO', None)
(6, 'reference_type', 'text', 'NO', None)
(7, 'reference_id', 'text', 'NO', None)
(8, 'notes', 'text', 'YES', None)
(9, 'created_date', 'timestamp without time zone', 'NO', None)
(10, 'created_by', 'text', 'NO', None)
(11, 'updated_date', 'timestamp without time zone', 'NO', None)
(12, 'updated_by', 'text', 'NO', None)
Inventory Movement table structure validation passed: True


## 6.4 Verify Keys, Relationships & Load Dependencies

Verify the Inventory Movement primary key, the approved Inventory foreign key, and the required parent records in `commerce.inventory`.

The known `INM026 → IN020` exception is expected to make the load dependency check fail. The notebook must not substitute another Inventory ID and must not delete the historical movement.


In [23]:
print("Inventory Movement Keys, Relationships & Load Dependency Review")
cursor.execute("""SELECT tc.constraint_name,kcu.column_name FROM information_schema.table_constraints AS tc JOIN information_schema.key_column_usage AS kcu ON tc.constraint_name=kcu.constraint_name AND tc.table_schema=kcu.table_schema WHERE tc.table_schema='commerce' AND tc.table_name='inventory_movement' AND tc.constraint_type='PRIMARY KEY';""")
primary_keys=cursor.fetchall()
primary_key_valid=len(primary_keys)==1 and primary_keys[0][1]=="inventory_movement_id"
print("Primary Key:",primary_keys)
print("Primary Key validation passed:",primary_key_valid)
cursor.execute("""SELECT tc.constraint_name,kcu.column_name,ccu.table_schema,ccu.table_name,ccu.column_name FROM information_schema.table_constraints AS tc JOIN information_schema.key_column_usage AS kcu ON tc.constraint_name=kcu.constraint_name AND tc.table_schema=kcu.table_schema JOIN information_schema.constraint_column_usage AS ccu ON ccu.constraint_name=tc.constraint_name AND ccu.table_schema=tc.table_schema WHERE tc.table_schema='commerce' AND tc.table_name='inventory_movement' AND tc.constraint_type='FOREIGN KEY' ORDER BY kcu.column_name;""")
foreign_keys=cursor.fetchall()
expected_foreign_keys={("inventory_id","inventory","inventory_id")}
actual_foreign_keys={(r[1],r[3],r[4]) for r in foreign_keys}
foreign_keys_valid=actual_foreign_keys==expected_foreign_keys
print("Foreign Keys:",foreign_keys)
print("Foreign Key validation passed:",foreign_keys_valid)
inventory_ids=inventory_movement_db_ready["inventory_id"].dropna().astype(str).unique().tolist()
placeholders=",".join(["%s"]*len(inventory_ids))
cursor.execute(f"SELECT inventory_id FROM commerce.inventory WHERE inventory_id IN ({placeholders});",inventory_ids)
existing_inventory_ids={str(r[0]) for r in cursor.fetchall()}
missing_inventory_ids_db=sorted(set(inventory_ids)-existing_inventory_ids)
inventory_dependency_valid=len(missing_inventory_ids_db)==0
print("Missing Inventory IDs in PostgreSQL:",missing_inventory_ids_db)
print("Inventory dependency ready:",inventory_dependency_valid)
load_dependency_ready=all([primary_key_valid,foreign_keys_valid,inventory_dependency_valid])
print("Inventory Movement is ready for data loading:",load_dependency_ready)


Inventory Movement Keys, Relationships & Load Dependency Review
Primary Key: [('inventory_movement_pkey', 'inventory_movement_id')]
Primary Key validation passed: True
Foreign Keys: [('inventory_movement_inventory_id_fkey', 'inventory_id', 'commerce', 'inventory', 'inventory_id')]
Foreign Key validation passed: True
Missing Inventory IDs in PostgreSQL: ['IN020']
Inventory dependency ready: False
Inventory Movement is ready for data loading: False


## 6.5 Insert Inventory Movement Records

Load the validated Inventory Movement Database-Ready dataset only after target structure and dependency checks pass.

Because `INM026` references unresolved `IN020`, insertion must be **blocked safely**. The notebook must not replace the parent ID, delete the movement, disable the foreign key, or partially load the dataset.


In [24]:
print("Inventory Movement Record Loading")
cursor.execute("SELECT COUNT(*) FROM commerce.inventory_movement;")
existing_record_count=cursor.fetchone()[0]
target_table_empty=existing_record_count==0
db_ready_row_count=len(inventory_movement_db_ready)
print("Existing records:",existing_record_count)
print("Target table is empty:",target_table_empty)
print("Database-ready records:",db_ready_row_count)
print("Load dependency ready:",load_dependency_ready)
if not load_dependency_ready:
    print("LOAD BLOCKED: unresolved Inventory parent dependency detected.")
    print("No records were inserted.")
    print("Known exception: INM026 references InventoryRecordID IN020, which is not present in commerce.inventory.")
else:
    insert_inventory_movement_query="""
    INSERT INTO commerce.inventory_movement
    (inventory_movement_id,inventory_id,movement_date,movement_type,quantity_change,reference_type,reference_id,notes,created_date,created_by,updated_date,updated_by)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s);
    """
    try:
        if not target_table_empty: raise ValueError("Target table is not empty. Insertion stopped to prevent accidental duplicate loading.")
        records_to_insert=[(row.inventory_movement_id,row.inventory_id,row.movement_date,row.movement_type,row.quantity_change,row.reference_type,row.reference_id,None,row.created_date,row.created_by,row.updated_date,row.updated_by) for row in inventory_movement_db_ready.itertuples(index=False)]
        cursor.executemany(insert_inventory_movement_query,records_to_insert)
        conn.commit()
        print("Inventory Movement records inserted:",len(records_to_insert))
        print("Transaction committed successfully.")
    except Exception as e:
        conn.rollback()
        print("Inventory Movement record insertion failed.")
        print("Transaction rolled back.")
        print("Error:",e)


Inventory Movement Record Loading
Existing records: 0
Target table is empty: True
Database-ready records: 26
Load dependency ready: False
LOAD BLOCKED: unresolved Inventory parent dependency detected.
No records were inserted.
Known exception: INM026 references InventoryRecordID IN020, which is not present in commerce.inventory.


## 6.6 Validate Inventory Movement Row Count

Compare the Database-Ready Inventory Movement row count with the PostgreSQL target row count.

If the load is blocked by an unresolved dependency, report the mismatch explicitly rather than asserting a false success.


In [25]:
db_ready_row_count=len(inventory_movement_db_ready)
cursor.execute("SELECT COUNT(*) FROM commerce.inventory_movement;")
postgresql_row_count=cursor.fetchone()[0]
row_count_match=db_ready_row_count==postgresql_row_count
print("Inventory Movement Row Count Validation")
print("Database-Ready Dataset Row Count:",db_ready_row_count)
print("PostgreSQL Table Row Count:",postgresql_row_count)
print("Row counts match:",row_count_match)
if load_dependency_ready:
    assert row_count_match
    print("Inventory Movement row count validation passed.")
else:
    print("Row-count validation is pending because database loading was correctly blocked by the unresolved parent dependency.")


Inventory Movement Row Count Validation
Database-Ready Dataset Row Count: 26
PostgreSQL Table Row Count: 0
Row counts match: False
Row-count validation is pending because database loading was correctly blocked by the unresolved parent dependency.


## 6.7 Retrieve Inventory Movement Records

Retrieve the loaded Inventory Movement records from PostgreSQL for direct review. If loading is blocked, the current target state is retrieved for review.


In [26]:
retrieve_inventory_movement_query="""
SELECT inventory_movement_id,inventory_id,movement_date,movement_type,quantity_change,reference_type,reference_id,notes,created_date,created_by,updated_date,updated_by
FROM commerce.inventory_movement ORDER BY inventory_movement_id;
"""
try:
    cursor.execute(retrieve_inventory_movement_query)
    inventory_movement_records=cursor.fetchall()
    column_names=[description[0] for description in cursor.description]
    inventory_movement_postgresql=pd.DataFrame(inventory_movement_records,columns=column_names)
    for column in ["movement_date","created_date","updated_date"]:
        inventory_movement_postgresql[column]=pd.to_datetime(inventory_movement_postgresql[column],errors="coerce")
    print("Inventory Movement Records Retrieved from PostgreSQL")
    print("Records retrieved:",len(inventory_movement_records))
    display(inventory_movement_postgresql)
except Exception as e:
    print("Inventory Movement record retrieval failed.")
    print("Error:",e)
    inventory_movement_postgresql=pd.DataFrame()


Inventory Movement Records Retrieved from PostgreSQL
Records retrieved: 0


,inventory_movement_id,inventory_id,movement_date,movement_type,quantity_change,reference_type,reference_id,notes,created_date,created_by,updated_date,updated_by


## 6.8 Source → Database Reconciliation

Compare the Inventory Movement Database-Ready dataset against the PostgreSQL target for row count, columns, primary keys, and record-level values.

A successful reconciliation is expected only after the database load dependency has been resolved and the dataset has been loaded.


In [27]:
if not load_dependency_ready:
    print("Inventory Movement Source → Database Reconciliation")
    print("Reconciliation not completed because database loading was blocked by the unresolved INM026 → IN020 dependency.")
    reconciliation_passed=False
else:
    source=inventory_movement_db_ready.copy().sort_values("inventory_movement_id").reset_index(drop=True)
    database=inventory_movement_postgresql.copy().sort_values("inventory_movement_id").reset_index(drop=True)
    for column in ["movement_date","created_date","updated_date"]:
        source[column]=pd.to_datetime(source[column],errors="coerce")
        database[column]=pd.to_datetime(database[column],errors="coerce")
    source_row_count=len(source)
    database_row_count=len(database)
    row_count_match=source_row_count==database_row_count
    source_columns=source.columns.tolist()
    database_columns=[c for c in database.columns if c!="notes"]
    column_match=source_columns==database_columns
    primary_key_match=set(source["inventory_movement_id"])==set(database["inventory_movement_id"])
    value_match=source.equals(database.drop(columns=["notes"]))
    reconciliation_passed=all([row_count_match,column_match,primary_key_match,value_match])
    print("Row counts match:",row_count_match)
    print("Mapped columns match:",column_match)
    print("Inventory Movement IDs match:",primary_key_match)
    print("All mapped record values match:",value_match)
    print("Source → Database reconciliation passed:",reconciliation_passed)
    assert reconciliation_passed


Inventory Movement Source → Database Reconciliation
Reconciliation not completed because database loading was blocked by the unresolved INM026 → IN020 dependency.


## 6.9 Inventory Movement Database Integrity Validation

Perform final Inventory Movement integrity checks covering required fields, primary key uniqueness, the Inventory foreign-key relationship, movement/reference values, and audit date relationships.

If loading is blocked, validate the current target state and explicitly report that final loaded-dataset integrity has not been achieved.


In [28]:
integrity_query="""
SELECT COUNT(*) AS total_rows,COUNT(DISTINCT inventory_movement_id) AS distinct_inventory_movement_ids,
COUNT(*) FILTER (WHERE inventory_movement_id IS NULL) AS null_inventory_movement_ids,
COUNT(*) FILTER (WHERE inventory_id IS NULL) AS null_inventory_ids,
COUNT(*) FILTER (WHERE movement_date IS NULL) AS null_movement_dates,
COUNT(*) FILTER (WHERE movement_type IS NULL) AS null_movement_types,
COUNT(*) FILTER (WHERE quantity_change IS NULL) AS null_quantity_changes,
COUNT(*) FILTER (WHERE reference_type IS NULL) AS null_reference_types,
COUNT(*) FILTER (WHERE reference_id IS NULL) AS null_reference_ids,
COUNT(*) FILTER (WHERE created_date IS NULL) AS null_created_dates,
COUNT(*) FILTER (WHERE created_by IS NULL) AS null_created_by,
COUNT(*) FILTER (WHERE updated_date IS NULL) AS null_updated_dates,
COUNT(*) FILTER (WHERE updated_by IS NULL) AS null_updated_by,
COUNT(*) FILTER (WHERE movement_type NOT IN ('Purchase','Sale','Return','Adjustment')) AS invalid_movement_types,
COUNT(*) FILTER (WHERE reference_type NOT IN ('Purchase Order','Sales Order','Return','Adjustment')) AS invalid_reference_types,
COUNT(*) FILTER (WHERE updated_date < created_date) AS invalid_audit_date_order
FROM commerce.inventory_movement;
"""
try:
    cursor.execute(integrity_query)
    r=cursor.fetchone()
    labels=["Total rows","Distinct inventory movement IDs","NULL inventory movement IDs","NULL inventory IDs","NULL movement dates","NULL movement types","NULL quantity changes","NULL reference types","NULL reference IDs","NULL created dates","NULL created by","NULL updated dates","NULL updated by","Invalid movement types","Invalid reference types","Updated before created"]
    for label,value in zip(labels,r): print(f"{label}: {value}")
    integrity_passed=(r[0]==r[1] and all(value==0 for value in r[2:]))
    print("Inventory Movement database integrity passed:",integrity_passed)
    if load_dependency_ready: assert integrity_passed
    else: print("Final loaded-dataset integrity is pending because the Inventory Movement load was blocked.")
except Exception as e:
    print("Inventory Movement database integrity validation failed.")
    print("Error:",e)


Total rows: 0
Distinct inventory movement IDs: 0
NULL inventory movement IDs: 0
NULL inventory IDs: 0
NULL movement dates: 0
NULL movement types: 0
NULL quantity changes: 0
NULL reference types: 0
NULL reference IDs: 0
NULL created dates: 0
NULL created by: 0
NULL updated dates: 0
NULL updated by: 0
Invalid movement types: 0
Invalid reference types: 0
Updated before created: 0
Inventory Movement database integrity passed: True
Final loaded-dataset integrity is pending because the Inventory Movement load was blocked.


## 6.10 Close PostgreSQL Connection

Close PostgreSQL resources cleanly after all Inventory Movement loading and validation steps are complete or after the load is safely blocked by an unresolved dependency.


In [29]:
try:
    if cursor is not None and not cursor.closed: cursor.close()
    if conn is not None and conn.closed == 0: conn.close()
    print("PostgreSQL resources closed successfully.")
    if cursor is not None: print("Cursor closed:",cursor.closed)
    if conn is not None: print("Connection closed:",conn.closed != 0)
except Exception as e:
    print("PostgreSQL resource closure failed.")
    print("Error:",e)


PostgreSQL resources closed successfully.
Cursor closed: True
Connection closed: True


# INVENTORY MOVEMENT ETL STATUS

**Notebook:** `13_inventory_movement_etl.ipynb`  
**Target:** `commerce.inventory_movement`

The notebook preserves the locked LJ Dev Commerce ETL workflow and the approved Inventory Movement source values.

**Known exception:** `INM026` references `IN020`, which is not present in the completed Inventory parent dataset. The notebook retains the source record unchanged and blocks PostgreSQL insertion until the parent relationship is resolved. No silent reassignment or deletion is performed.

**Execution status:** Notebook generated and statically validated only. Runtime execution must be performed in the project environment.
